# Which sellers should a limited budget go to first? (Phase 5)

Phase 4 found sellers whose on-time orders are unhappy more often than usual, beyond chance. Olist
can't help every seller at once. This notebook uses **linear programming** (PuLP) to choose which
sellers to include in a pilot under a fixed budget. The data has no costs, so every cost and benefit
is a written-down **assumption**. The output is a ranked list **worth testing**, not a promise of results.

**Contents:** 1. Setup · 2. A tiny example

## 1. Setup
**PuLP** lets us describe a choice problem in plain Python: what we can choose, what we want to
maximise, and the limits. A **solver** (here HiGHS, free and open source) then finds the best answer.

In [1]:
import pulp
print("PuLP", pulp.__version__, "| solvers available:", pulp.listSolvers(onlyAvailable=True))

PuLP 4.2.0 | solvers available: ['HiGHS']


## 2. A tiny example (made-up numbers)
Four sellers. Helping each one costs something and is expected to avoid some unhappy orders.
The budget is 10. Which sellers should we pick to avoid the most unhappy orders?

In [2]:
toy = {            # seller: (cost, unhappy orders avoided)  - made-up numbers for the example
    "A": (6, 30),
    "B": (5, 20),
    "C": (5, 20),
    "D": (3, 9),
}
BUDGET = 10

model = pulp.LpProblem("toy", pulp.LpMaximize)                        # we want the biggest benefit
pick = {s: model.add_variable(f"pick_{s}", cat="Binary") for s in toy}  # 1 = help this seller, 0 = don't
model += pulp.lpSum(toy[s][1] * pick[s] for s in toy)                 # goal: total unhappy orders avoided
model += pulp.lpSum(toy[s][0] * pick[s] for s in toy) <= BUDGET       # limit: total cost within budget
result = model.solve(pulp.HiGHS(msg=False))                           # the solver does the maths

chosen = [s for s in toy if pick[s].value() > 0.5]
print("status:", result.status.name, "| chosen:", chosen,
      "| cost:", sum(toy[s][0] for s in chosen), "| avoided:", sum(toy[s][1] for s in chosen))

status: Optimal | chosen: ['B', 'C'] | cost: 10 | avoided: 40


**Why not just take the "best" seller first?** Seller A has the biggest benefit, so a simple
"best first" rule takes A (cost 6), then only D still fits (cost 3): 39 avoided. The model finds
B + C (cost 10): 40 avoided. With a budget, the best *combination* is not always the best items
one by one, and that is the job PuLP does for us.